**To download data from the year 2000, kaggle had a dataset.**

[Check this out](https://www.kaggle.com/datasets/ujjvalpatel1003/nse-historical-data-1990-2023)

Data is adjusted for splits and bonuses.

#### Indicator calculation and data pre processing

In [ ]:
import glob
import os
import pandas as pd

# 2. Set Paths
KAGGLE_DATA_DIR = '/content/drive/MyDrive/data till 2024-DEC/data till 2024-DEC/data'
OUTPUT_DIR = '/content/drive/MyDrive/Stock_Backtest/1_weekly_universes/'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_and_preprocess_weekly(data_dir):
  """Reads all CSVs ONCE and resamples them to weekly candles to optimize performance."""
  all_files = glob.glob(os.path.join(data_dir, '*.csv'))
  print(f'Found {len(all_files)} files. Pre-processing into weekly data...')

  weekly_data_list = []

  for file in all_files:
    ticker = os.path.basename(file).replace('.csv', '')
    try:
      df = pd.read_csv(file)
      df['datetime'] = pd.to_datetime(df['datetime'])

      # Set index for resampling
      df.set_index('datetime', inplace=True)

      # Resample to weekly
      df_weekly = (
          df.resample('W-SUN')
          .agg({
              'open': 'first',
              'high': 'max',
              'low': 'min',
              'close': 'last',
              'volume': 'sum',
          })
          .dropna()
      )

      # Calculate weekly turnover (Close * Volume) for universe ranking
      df_weekly['turnover'] = df_weekly['close'] * df_weekly['volume']
      df_weekly['Ticker'] = ticker
      weekly_data_list.append(df_weekly.reset_index())

    except Exception:
      continue

  combined_df = pd.concat(weekly_data_list, ignore_index=True)
  combined_df.rename(columns={'datetime': 'Date'}, inplace=True)
  return combined_df


def process_2year_blocks(
    df_all_weekly, start_year=2017, end_year=2024, top_n=500
):
  for active_start in range(start_year, end_year, 2):
    active_end = active_start + 1
    buffer_start = active_start - 1  # 1-year lookback buffer

    print(
        f'\n--- Processing Block: Trading {active_start}-{active_end}'
        f' (Universe Selected via {buffer_start}) ---'
    )

    # 1. NO LOOKAHEAD BIAS: Determine Top N liquid stocks strictly using the BUFFER year
    buffer_mask = df_all_weekly['Date'].dt.year == buffer_start
    df_buffer = df_all_weekly[buffer_mask]

    if df_buffer.empty:
      print(f'No buffer data found for year {buffer_start}. Skipping...')
      continue

    top_tickers = (
        df_buffer.groupby('Ticker')['turnover']
        .sum()
        .nlargest(top_n)
        .index.tolist()
    )

    # 2. Extract FULL 3-year data window (1-yr buffer + 2-yr trading) for top tickers
    full_mask = (df_all_weekly['Date'].dt.year >= buffer_start) & (
        df_all_weekly['Date'].dt.year <= active_end
    )
    block_universe = df_all_weekly[
        full_mask & df_all_weekly['Ticker'].isin(top_tickers)
    ].copy()

    # Drop turnover temporary column and format
    block_universe.drop(columns=['turnover'], inplace=True, errors='ignore')
    block_universe.rename(
        columns={
            'open': 'Open',
            'high': 'High',
            'low': 'Low',
            'close': 'Close',
            'volume': 'Volume',
        },
        inplace=True,
    )

    # Sort clearly by Ticker and Date
    block_universe.sort_values(by=['Ticker', 'Date'], inplace=True)

    # 3. Save to Parquet
    out_file = os.path.join(
        OUTPUT_DIR, f'universe_{active_start}_{active_end}.parquet'
    )
    block_universe.to_parquet(out_file, index=False)
    print(
        f'Saved {len(top_tickers)} stocks across {buffer_start}-{active_end} to'
        f' {out_file}'
    )


# --- Execution ---
# Step 1: Preprocess all data into memory ONCE
df_all_weekly = load_and_preprocess_weekly(KAGGLE_DATA_DIR)

# Step 2: Generate 2-year blocks (2000-2001, 2002-2003, etc.)
process_2year_blocks(df_all_weekly, start_year=2017, end_year=2024, top_n=500)

Found 2505 files. Pre-processing into weekly data...

--- Processing Block: Trading 2017-2018 (Universe Selected via 2016) ---
Saved 500 stocks across 2016-2018 to /content/drive/MyDrive/Stock_Backtest/1_weekly_universes/universe_2017_2018.parquet

--- Processing Block: Trading 2019-2020 (Universe Selected via 2018) ---
Saved 500 stocks across 2018-2020 to /content/drive/MyDrive/Stock_Backtest/1_weekly_universes/universe_2019_2020.parquet

--- Processing Block: Trading 2021-2022 (Universe Selected via 2020) ---
Saved 500 stocks across 2020-2022 to /content/drive/MyDrive/Stock_Backtest/1_weekly_universes/universe_2021_2022.parquet

--- Processing Block: Trading 2023-2024 (Universe Selected via 2022) ---
Saved 500 stocks across 2022-2024 to /content/drive/MyDrive/Stock_Backtest/1_weekly_universes/universe_2023_2024.parquet


In [ ]:
import glob
import os
import numpy as np
import pandas as pd

# 1. Set Input and Output Paths
INPUT_DIR = '/content/drive/MyDrive/Stock_Backtest/1_weekly_universes/'
OUTPUT_DIR = '/content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/'
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 2. Indicator Functions
def calculate_rsi(series, period=14):
  delta = series.diff()
  gain = (delta.where(delta > 0, 0)).ewm(alpha=1 / period, adjust=False).mean()
  loss = (-delta.where(delta < 0, 0)).ewm(alpha=1 / period, adjust=False).mean()
  rs = gain / loss
  return 100 - (100 / (1 + rs))


def calculate_supertrend(df, period=8, multiplier=2.5):
    high, low, close = df['High'], df['Low'], df['Close']

    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1 / period, adjust=False).mean()

    hl2 = (high + low) / 2
    basic_upper = hl2 + (multiplier * atr)
    basic_lower = hl2 - (multiplier * atr)

    final_upper = basic_upper.copy()
    final_lower = basic_lower.copy()
    supertrend = np.zeros(len(df))
    direction = np.zeros(len(df))

    # Convert to explicit writeable numpy arrays
    close_arr = close.to_numpy()
    b_upper_arr = basic_upper.to_numpy()
    b_lower_arr = basic_lower.to_numpy()

    f_upper_arr = final_upper.to_numpy(copy=True)
    f_lower_arr = final_lower.to_numpy(copy=True)

    for i in range(1, len(df)):
        if (b_upper_arr[i] < f_upper_arr[i - 1]) or (
            close_arr[i - 1] > f_upper_arr[i - 1]
        ):
            f_upper_arr[i] = b_upper_arr[i]
        else:
            f_upper_arr[i] = f_upper_arr[i - 1]

        if (b_lower_arr[i] > f_lower_arr[i - 1]) or (
            close_arr[i - 1] < f_lower_arr[i - 1]
        ):
            f_lower_arr[i] = b_lower_arr[i]
        else:
            f_lower_arr[i] = f_lower_arr[i - 1]

        if direction[i - 1] == 1:
            if close_arr[i] < f_lower_arr[i]:
                direction[i] = -1
                supertrend[i] = f_upper_arr[i]
            else:
                direction[i] = 1
                supertrend[i] = f_lower_arr[i]
        else:
            if close_arr[i] > f_upper_arr[i]:
                direction[i] = 1
                supertrend[i] = f_lower_arr[i]
            else:
                direction[i] = -1
                supertrend[i] = f_upper_arr[i]

    return pd.Series(supertrend, index=df.index), pd.Series(
        direction, index=df.index
    )


def calculate_raw_rs(series):
  """Calculates weighted return across 1, 3, 6, and 9 months."""
  r5 = series.pct_change(5)
  r13 = series.pct_change(13)
  r26 = series.pct_change(26)
  r39 = series.pct_change(39)
  return ((0.3 * r5) + (0.3 * r13) + (0.2 * r26) + (0.2 * r39))


# 3. Process Block Files
def process_indicator_blocks():
  universe_files = sorted(glob.glob(os.path.join(INPUT_DIR, '*.parquet')))

  for file_path in universe_files:
    file_name = os.path.basename(file_path)
    years = [
        int(s) for s in file_name.replace('.parquet', '').split('_') if s.isdigit()
    ]
    active_start, active_end = years[0], years[1]

    print(f'Processing indicators for Block {active_start}-{active_end}...')

    df = pd.read_parquet(file_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df.sort_values(by=['Ticker', 'Date'], inplace=True)

    processed_dfs = []

    # Step A: Compute technical indicators per Ticker
    for ticker, group in df.groupby('Ticker'):
      group = group.copy()
      group['RSI_14'] = calculate_rsi(group['Close'], period=14)
      group['Supertrend'], group['Supertrend_Direction'] = (
          calculate_supertrend(group, period=8, multiplier=2.5)
      )
      group['RS_Raw'] = calculate_raw_rs(group['Close'])
      processed_dfs.append(group)

    combined_df = pd.concat(processed_dfs, ignore_index=True)

    # Step B: CROSS-SECTIONAL NORMALIZATION (0-100 Percentile Rank per Date)
    combined_df['RS_Score'] = (
        combined_df.groupby('Date')['RS_Raw'].rank(pct=True) * 100
    )
    combined_df.drop(columns=['RS_Raw'], inplace=True)

    # Step C: Filter out lookback buffer year; retain active trading period
    active_mask = (combined_df['Date'].dt.year >= active_start) & (
        combined_df['Date'].dt.year <= active_end
    )
    final_df = combined_df[active_mask].copy()

    # Save to Output Folder
    out_file = os.path.join(
        OUTPUT_DIR, f'indicators_{active_start}_{active_end}.parquet'
    )
    final_df.to_parquet(out_file, index=False)
    print(f'Saved: {out_file} (Rows: {len(final_df)})')


process_indicator_blocks()

Processing indicators for Block 2017-2018...
Saved: /content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/indicators_2017_2018.parquet (Rows: 52494)
Processing indicators for Block 2019-2020...
Saved: /content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/indicators_2019_2020.parquet (Rows: 51963)
Processing indicators for Block 2021-2022...
Saved: /content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/indicators_2021_2022.parquet (Rows: 52000)
Processing indicators for Block 2023-2024...
Saved: /content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/indicators_2023_2024.parquet (Rows: 52462)


#### Backtest

In [ ]:
import glob
import math
import os
import pandas as pd

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
INDICATORS_DIR = '/content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/'
OUTPUT_DIR = '/content/drive/MyDrive/Stock_Backtest/3_backtest_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

INITIAL_CAPITAL = 10_00_000.0  # ₹10 Lakhs
MAX_POSITIONS = 15
RISK_FREE_RATE_ANNUAL = 0.00  # 0.0% or set to 0.06 for 6% G-Sec rate

# Market Realism Assumptions
SLIPPAGE_PCT = 0.0000  # e.g., 0.0010 (0.10%)
TRANSACTION_FEE_PCT = 0.005  # 0.50% per trade (Brokerage, STT, Exchange fees)

TRADES_LOG_OUTPUT = os.path.join(OUTPUT_DIR, 'completed_trades_log.csv')
EQUITY_CURVE_OUTPUT = os.path.join(OUTPUT_DIR, 'equity_curve.csv')

# ==========================================
# MULTI-BLOCK BACKTEST ENGINE
# ==========================================
indicator_files = sorted(glob.glob(os.path.join(INDICATORS_DIR, '*.parquet')))

if not indicator_files:
  raise FileNotFoundError(
      f'No parquet indicator files found in {INDICATORS_DIR}'
  )

current_capital = INITIAL_CAPITAL
completed_trades_log = []
equity_history = []
dates_history = []
total_brokerage_paid = 0.0

print(f'Starting Backtest across {len(indicator_files)} 2-year blocks...\n')

for file_path in indicator_files:
  file_name = os.path.basename(file_path)
  years = [
      int(s)
      for s in file_name.replace('.parquet', '').split('_')
      if s.isdigit()
  ]
  active_start, active_end = years[0], years[1]

  print(
      f'--- Running Block {active_start}-{active_end} | Starting Cash:'
      f' Rs.{current_capital:,.2f} ---'
  )

  # Load and standardize schema
  weekly_df = pd.read_parquet(file_path)
  weekly_df['Date'] = pd.to_datetime(weekly_df['Date'])

  # Map column names from indicator generator
  column_mapping = {
      'RSI_14': 'RSI',
      'Supertrend_Direction': 'ST_Direction',
      'RS_Score': 'RS_Percentile',
  }
  weekly_df.rename(columns=column_mapping, inplace=True)

  # Remove duplicate ticker entries per date
  weekly_df = weekly_df.drop_duplicates(subset=['Date', 'Ticker']).copy()

  sorted_dates = sorted(weekly_df['Date'].unique())

  positions = {}
  pending_entries = []
  pending_exits = []
  prev_week_closes = {}
  cash = current_capital

  for idx, current_date in enumerate(sorted_dates):
    week_data = weekly_df[weekly_df['Date'] == current_date].set_index('Ticker')
    if week_data.empty:
      continue

    # -------------------------------------------------------------------
    # STEP 1: EXECUTE EXITS AT MONDAY OPEN
    # -------------------------------------------------------------------
    for exit_item in pending_exits:
      ticker = exit_item['ticker']
      if ticker in positions and ticker in week_data.index:
        pos = positions[ticker]
        row = week_data.loc[ticker]
        raw_open = row['Open']

        exec_sell_price = raw_open * (1 - SLIPPAGE_PCT)
        gross_proceeds = pos['units'] * exec_sell_price

        raw_trade_value = pos['units'] * raw_open
        sell_brokerage = raw_trade_value * TRANSACTION_FEE_PCT
        net_proceeds = gross_proceeds - sell_brokerage

        cash += net_proceeds
        total_brokerage_paid += sell_brokerage

        net_pnl_pct = (
            (net_proceeds - pos['total_cost']) / pos['total_cost']
        ) * 100

        completed_trades_log.append({
            'Block': f'{active_start}-{active_end}',
            'Ticker': ticker,
            'Signal_Date': pos['signal_date'].strftime('%Y-%m-%d'),
            'Buy_Date': pos['buy_date'].strftime('%Y-%m-%d'),
            'Buy_Price': round(pos['buy_price'], 2),
            'Units': pos['units'],
            'Entry_RSI': round(pos['rsi_signal'], 2),
            'Entry_ST': pos['st_signal'],
            'Signal_RS_Score': round(pos['rs_signal'], 2),
            'Exit_Signal_Date': exit_item['exit_signal_date'].strftime(
                '%Y-%m-%d'
            ),
            'Sell_Date': current_date.strftime('%Y-%m-%d'),
            'Sell_Price': round(exec_sell_price, 2),
            'Exit_RSI': round(exit_item['exit_rsi'], 2),
            'Exit_ST': exit_item['exit_st'],
            'Return_%': round(net_pnl_pct, 2),
            'Exit_Reason': 'Strategy Exit Signal',
        })

        del positions[ticker]

    pending_exits = []  # Reset after execution

    # -------------------------------------------------------------------
    # STEP 2: EXECUTE BUYS AT MONDAY OPEN
    # -------------------------------------------------------------------
    invested_at_monday_open = sum(
        pos['units'] * prev_week_closes.get(tkr, pos['buy_price'])
        for tkr, pos in positions.items()
    )
    monday_total_equity = cash + invested_at_monday_open
    available_slots = MAX_POSITIONS - len(positions)

    if pending_entries and available_slots > 0:
      all_valid_candidates = [
          item
          for item in pending_entries
          if item['ticker'] in week_data.index
          and item['ticker'] not in positions
      ]
      valid_candidates = all_valid_candidates[:available_slots]
      num_trades = len(valid_candidates)

      if num_trades > 0 and cash > 0:
        target_slot_capital = monday_total_equity / MAX_POSITIONS
        capital_per_trade = min(target_slot_capital, cash / num_trades)

        for item in valid_candidates:
          ticker = item['ticker']
          row = week_data.loc[ticker]
          raw_open = row['Open']

          exec_buy_price = raw_open * (1 + SLIPPAGE_PCT)
          effective_cost_per_unit = exec_buy_price + (
              raw_open * TRANSACTION_FEE_PCT
          )

          if effective_cost_per_unit > 0:
            alloc = min(capital_per_trade, cash)
            units = math.floor(alloc / effective_cost_per_unit)

            if units > 0:
              gross_buy_cost = units * exec_buy_price
              raw_trade_value = units * raw_open
              buy_brokerage = raw_trade_value * TRANSACTION_FEE_PCT
              actual_cost = gross_buy_cost + buy_brokerage

              if actual_cost <= cash:
                cash -= actual_cost
                total_brokerage_paid += buy_brokerage

                positions[ticker] = {
                    'units': units,
                    'buy_price': exec_buy_price,
                    'total_cost': actual_cost,
                    'buy_date': current_date,
                    'signal_date': item['signal_date'],
                    'rsi_signal': item['rsi_signal'],
                    'st_signal': item['st_signal'],
                    'rs_signal': item['rs_signal'],
                }

    pending_entries = []  # Reset outside conditional (Prevents stale signals)

    # -------------------------------------------------------------------
    # STEP 3: SCAN SIGNALS AT FRIDAY CLOSE (For Next Week Execution)
    # -------------------------------------------------------------------
    # Don't scan new entries on the final week of the block (liquidating)
    if idx < len(sorted_dates) - 1:
      for ticker, pos in positions.items():
        if ticker in week_data.index:
          row = week_data.loc[ticker]
          if row['RSI'] < 40 or row['ST_Direction'] == -1:
            pending_exits.append({
                'ticker': ticker,
                'exit_signal_date': current_date,
                'exit_rsi': row['RSI'],
                'exit_st': row['ST_Direction'],
            })

      tickers_exiting_next = [x['ticker'] for x in pending_exits]
      estimated_free_slots = MAX_POSITIONS - (
          len(positions) - len(pending_exits)
      )

      if estimated_free_slots > 0:
        candidates = week_data[
            (week_data['RSI'] > 60)
            & (week_data['ST_Direction'] == 1)
            & (week_data['RS_Percentile'] >= 92)
            #& (week_data['RS_Percentile'] <= 99)
            & (~week_data.index.isin(positions.keys()))
            & (~week_data.index.isin(tickers_exiting_next))
        ]

        if not candidates.empty:
          sorted_candidates = candidates.sort_values(
              by=['RS_Percentile', 'RSI'], ascending=[False, False]
          )
          for tkr, row in sorted_candidates.iterrows():
            pending_entries.append({
                'ticker': tkr,
                'signal_date': current_date,
                'rsi_signal': row['RSI'],
                'st_signal': row['ST_Direction'],
                'rs_signal': row['RS_Percentile'],
            })

    # -------------------------------------------------------------------
    # STEP 4: RECORD END-OF-WEEK EQUITY
    # -------------------------------------------------------------------
    friday_invested_value = sum(
        pos['units']
        * (
            week_data.loc[tkr, 'Close']
            if tkr in week_data.index
            else prev_week_closes.get(tkr, pos['buy_price'])
        )
        for tkr, pos in positions.items()
    )
    equity_history.append(cash + friday_invested_value)
    dates_history.append(current_date)

    for tkr in week_data.index:
      prev_week_closes[tkr] = week_data.loc[tkr, 'Close']

  # -------------------------------------------------------------------
  # BLOCK END: FORCE LIQUIDATE ALL OPEN POSITIONS TO CASH
  # -------------------------------------------------------------------
  final_week_data = weekly_df[
      weekly_df['Date'] == sorted_dates[-1]
  ].set_index('Ticker')

  for tkr, pos in list(positions.items()):
    last_close = (
        final_week_data.loc[tkr, 'Close']
        if tkr in final_week_data.index
        else pos['buy_price']
    )
    exec_sell_price = last_close * (1 - SLIPPAGE_PCT)
    gross_proceeds = pos['units'] * exec_sell_price

    raw_trade_value = pos['units'] * last_close
    sell_brokerage = raw_trade_value * TRANSACTION_FEE_PCT
    net_proceeds = gross_proceeds - sell_brokerage

    cash += net_proceeds
    total_brokerage_paid += sell_brokerage

    net_pnl_pct = (
        (net_proceeds - pos['total_cost']) / pos['total_cost']
    ) * 100

    completed_trades_log.append({
        'Block': f'{active_start}-{active_end}',
        'Ticker': tkr,
        'Signal_Date': pos['signal_date'].strftime('%Y-%m-%d'),
        'Buy_Date': pos['buy_date'].strftime('%Y-%m-%d'),
        'Buy_Price': round(pos['buy_price'], 2),
        'Units': pos['units'],
        'Entry_RSI': round(pos['rsi_signal'], 2),
        'Entry_ST': pos['st_signal'],
        'Signal_RS_Score': round(pos['rs_signal'], 2),
        'Exit_Signal_Date': sorted_dates[-1].strftime('%Y-%m-%d'),
        'Sell_Date': sorted_dates[-1].strftime('%Y-%m-%d'),
        'Sell_Price': round(exec_sell_price, 2),
        'Exit_RSI': 0.0,
        'Exit_ST': 0,
        'Return_%': round(net_pnl_pct, 2),
        'Exit_Reason': '2-Year Block Forced Liquidation',
    })

  positions.clear()
  current_capital = cash
  print(
      f'Block {active_start}-{active_end} Finished | Ending Cash:'
      f' Rs.{current_capital:,.2f}\n'
  )

# ==========================================
# EXPORT LOGS AND PERFORMANCE SUMMARY
# ==========================================
trades_df = pd.DataFrame(completed_trades_log)
portfolio_df = pd.DataFrame({'Date': dates_history, 'Equity': equity_history})

trades_df.to_csv(TRADES_LOG_OUTPUT, index=False)
portfolio_df.to_csv(EQUITY_CURVE_OUTPUT, index=False)

if not portfolio_df.empty:
  final_capital = portfolio_df['Equity'].iloc[-1]
  total_return_pct = (
      (final_capital - INITIAL_CAPITAL) / INITIAL_CAPITAL
  ) * 100

  start_date = pd.to_datetime(portfolio_df['Date'].iloc[0])
  end_date = pd.to_datetime(portfolio_df['Date'].iloc[-1])
  total_years = (end_date - start_date).days / 365.25

  cagr_pct = (
      (((final_capital / INITIAL_CAPITAL) ** (1 / total_years)) - 1) * 100
      if total_years > 0
      else 0.0
  )
  avg_annual_brokerage = (
      total_brokerage_paid / total_years if total_years > 0 else 0.0
  )

  # Global Max Drawdown Calculation
  cum_max = portfolio_df['Equity'].cummax()
  portfolio_df['Drawdown_%'] = (
      (portfolio_df['Equity'] - cum_max) / cum_max
  ) * 100
  max_drawdown_pct = portfolio_df['Drawdown_%'].min()

  # Sharpe Ratio Calculation (Annualized Weekly)
  portfolio_df['Weekly_Return'] = portfolio_df['Equity'].pct_change()
  weekly_returns = portfolio_df['Weekly_Return'].dropna()

  rf_weekly = (1 + RISK_FREE_RATE_ANNUAL) ** (1 / 52) - 1
  excess_weekly_returns = weekly_returns - rf_weekly

  std_weekly = weekly_returns.std(ddof=1)
  sharpe_ratio = (
      (excess_weekly_returns.mean() / std_weekly) * math.sqrt(52)
      if std_weekly > 0
      else 0.0
  )

  total_trades = len(trades_df)
  win_rate_pct = (
      (len(trades_df[trades_df['Return_%'] > 0]) / total_trades) * 100
      if total_trades > 0
      else 0.0
  )

  portfolio_df['Year'] = pd.to_datetime(portfolio_df['Date']).dt.year
  yearly_returns = []
  yearly_drawdowns = []
  unique_years = portfolio_df['Year'].unique()

  for i, yr in enumerate(unique_years):
    group = portfolio_df[portfolio_df['Year'] == yr]
    start_val = (
        INITIAL_CAPITAL
        if i == 0
        else portfolio_df[portfolio_df['Year'] == unique_years[i - 1]][
            'Equity'
        ].iloc[-1]
    )
    end_val = group['Equity'].iloc[-1]

    y_return = ((end_val - start_val) / start_val) * 100
    yearly_returns.append(y_return)
    yearly_drawdowns.append(group['Drawdown_%'].min())

  avg_annual_return_pct = pd.Series(yearly_returns).mean()
  avg_annual_drawdown_pct = pd.Series(yearly_drawdowns).mean()

  print('=' * 60)
  print('          MULTI-BLOCK OVERALL STRATEGY PERFORMANCE')
  print('=' * 60)
  print(f'Initial Capital        : Rs.{INITIAL_CAPITAL:,.2f}')
  print(f'Final Capital          : Rs.{final_capital:,.2f}')
  print(f'Total Return           : {total_return_pct:+.2f}%')
  print(f'CAGR                   : {cagr_pct:.2f}%')
  print(
      'Sharpe Ratio'
      f' (R_f={RISK_FREE_RATE_ANNUAL*100:.1f}%): {sharpe_ratio:.2f}'
  )
  print(f'Max Drawdown           : {max_drawdown_pct:.2f}%')
  print(f'Total Trades           : {total_trades}')
  print(f'Win Rate               : {win_rate_pct:.2f}%')
  print(f'Total Brokerage Paid   : Rs.{total_brokerage_paid:,.2f}')
  print(f'Avg Annual Brokerage   : Rs.{avg_annual_brokerage:,.2f}')
  print(f'Avg Annual Return      : {avg_annual_return_pct:+.2f}%')
  print(f'Avg Annual Drawdown    : {avg_annual_drawdown_pct:.2f}%')
  print('=' * 60)

Starting Backtest across 4 2-year blocks...

--- Running Block 2017-2018 | Starting Cash: Rs.1,000,000.00 ---
Block 2017-2018 Finished | Ending Cash: Rs.1,156,665.98

--- Running Block 2019-2020 | Starting Cash: Rs.1,156,665.98 ---
Block 2019-2020 Finished | Ending Cash: Rs.1,146,881.22

--- Running Block 2021-2022 | Starting Cash: Rs.1,146,881.22 ---
Block 2021-2022 Finished | Ending Cash: Rs.2,170,943.68

--- Running Block 2023-2024 | Starting Cash: Rs.2,170,943.68 ---
Block 2023-2024 Finished | Ending Cash: Rs.5,767,011.41

          MULTI-BLOCK OVERALL STRATEGY PERFORMANCE
Initial Capital        : Rs.1,000,000.00
Final Capital          : Rs.5,795,724.60
Total Return           : +479.57%
CAGR                   : 24.59%
Sharpe Ratio (R_f=0.0%): 1.00
Max Drawdown           : -49.75%
Total Trades           : 223
Win Rate               : 42.15%
Total Brokerage Paid   : Rs.221,751.86
Avg Annual Brokerage   : Rs.27,747.47
Avg Annual Return      : +28.32%
Avg Annual Drawdown    : -28.30%


With nifty50 200day sma filter

In [ ]:
import pandas as pd
import yfinance as yf

# 1. Download Nifty 50 daily data
nifty_daily = yf.download('^BSESN', start='1999-01-01', progress=False)

# Flatten MultiIndex if present
if isinstance(nifty_daily.columns, pd.MultiIndex):
    nifty_daily.columns = nifty_daily.columns.droplevel(1)

# Clean close series & calculate SMA200
close_series = nifty_daily['Close'].squeeze()
sma200 = close_series.rolling(window=200).mean()
nifty_bullish = close_series > sma200

# 2. Normalize index to remove timezone/time components
nifty_bullish.index = pd.to_datetime(nifty_bullish.index).tz_localize(None).normalize()

# 3. Create full daily calendar range & forward-fill missing dates (holidays/weekends)
full_calendar = pd.date_range(start='1999-01-01', end=nifty_bullish.index.max(), freq='D')
nifty_bullish_ffill = nifty_bullish.reindex(full_calendar).ffill().fillna(False)

# 4. Create lookup dictionary (Key: Timestamp -> Value: True/False)
nifty_regime_dict = nifty_bullish_ffill.to_dict()

/tmp/ipykernel_2351/938221538.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty_daily = yf.download('^BSESN', start='1999-01-01', progress=False)
/tmp/ipykernel_2351/938221538.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  nifty_bullish_ffill = nifty_bullish.reindex(full_calendar).ffill().fillna(False)


In [ ]:
import glob
import math
import os
import pandas as pd

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
INDICATORS_DIR = '/content/drive/MyDrive/Stock_Backtest/2_weekly_indicators/'
OUTPUT_DIR = '/content/drive/MyDrive/Stock_Backtest/3_backtest_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

INITIAL_CAPITAL = 10_00_000.0  # ₹10 Lakhs
MAX_POSITIONS = 15
RISK_FREE_RATE_ANNUAL = 0.00  # 0.0% or set to 0.06 for 6% G-Sec rate

# Market Realism Assumptions
SLIPPAGE_PCT = 0.0000  # e.g., 0.0010 (0.10%)
TRANSACTION_FEE_PCT = 0.005  # 0.50% per trade (Brokerage, STT, Exchange fees)

TRADES_LOG_OUTPUT = os.path.join(OUTPUT_DIR, 'completed_trades_log.csv')
EQUITY_CURVE_OUTPUT = os.path.join(OUTPUT_DIR, 'equity_curve.csv')

# ==========================================
# MULTI-BLOCK BACKTEST ENGINE
# ==========================================
indicator_files = sorted(glob.glob(os.path.join(INDICATORS_DIR, '*.parquet')))

if not indicator_files:
  raise FileNotFoundError(
      f'No parquet indicator files found in {INDICATORS_DIR}'
  )

current_capital = INITIAL_CAPITAL
completed_trades_log = []
equity_history = []
dates_history = []
total_brokerage_paid = 0.0

print(f'Starting Backtest across {len(indicator_files)} 2-year blocks...\n')

for file_path in indicator_files:
  file_name = os.path.basename(file_path)
  years = [
      int(s)
      for s in file_name.replace('.parquet', '').split('_')
      if s.isdigit()
  ]
  active_start, active_end = years[0], years[1]

  print(
      f'--- Running Block {active_start}-{active_end} | Starting Cash:'
      f' Rs.{current_capital:,.2f} ---'
  )

  # Load and standardize schema
  weekly_df = pd.read_parquet(file_path)
  weekly_df['Date'] = pd.to_datetime(weekly_df['Date'])

  # Map column names from indicator generator
  column_mapping = {
      'RSI_14': 'RSI',
      'Supertrend_Direction': 'ST_Direction',
      'RS_Score': 'RS_Percentile',
  }
  weekly_df.rename(columns=column_mapping, inplace=True)

  # Remove duplicate ticker entries per date
  weekly_df = weekly_df.drop_duplicates(subset=['Date', 'Ticker']).copy()

  sorted_dates = sorted(weekly_df['Date'].unique())

  positions = {}
  pending_entries = []
  pending_exits = []
  prev_week_closes = {}
  cash = current_capital

  for idx, current_date in enumerate(sorted_dates):
    week_data = weekly_df[weekly_df['Date'] == current_date].set_index('Ticker')
    if week_data.empty:
      continue

    # -------------------------------------------------------------------
    # STEP 1: EXECUTE EXITS AT MONDAY OPEN
    # -------------------------------------------------------------------
    for exit_item in pending_exits:
      ticker = exit_item['ticker']
      if ticker in positions and ticker in week_data.index:
        pos = positions[ticker]
        row = week_data.loc[ticker]
        raw_open = row['Open']

        exec_sell_price = raw_open * (1 - SLIPPAGE_PCT)
        gross_proceeds = pos['units'] * exec_sell_price

        raw_trade_value = pos['units'] * raw_open
        sell_brokerage = raw_trade_value * TRANSACTION_FEE_PCT
        net_proceeds = gross_proceeds - sell_brokerage

        cash += net_proceeds
        total_brokerage_paid += sell_brokerage

        net_pnl_pct = (
            (net_proceeds - pos['total_cost']) / pos['total_cost']
        ) * 100

        completed_trades_log.append({
            'Block': f'{active_start}-{active_end}',
            'Ticker': ticker,
            'Signal_Date': pos['signal_date'].strftime('%Y-%m-%d'),
            'Buy_Date': pos['buy_date'].strftime('%Y-%m-%d'),
            'Buy_Price': round(pos['buy_price'], 2),
            'Units': pos['units'],
            'Entry_RSI': round(pos['rsi_signal'], 2),
            'Entry_ST': pos['st_signal'],
            'Signal_RS_Score': round(pos['rs_signal'], 2),
            'Exit_Signal_Date': exit_item['exit_signal_date'].strftime(
                '%Y-%m-%d'
            ),
            'Sell_Date': current_date.strftime('%Y-%m-%d'),
            'Sell_Price': round(exec_sell_price, 2),
            'Exit_RSI': round(exit_item['exit_rsi'], 2),
            'Exit_ST': exit_item['exit_st'],
            'Return_%': round(net_pnl_pct, 2),
            'Exit_Reason': 'Strategy Exit Signal',
        })

        del positions[ticker]

    pending_exits = []  # Reset after execution

    # -------------------------------------------------------------------
    # STEP 2: EXECUTE BUYS AT MONDAY OPEN
    # -------------------------------------------------------------------
    invested_at_monday_open = sum(
        pos['units'] * prev_week_closes.get(tkr, pos['buy_price'])
        for tkr, pos in positions.items()
    )
    monday_total_equity = cash + invested_at_monday_open
    available_slots = MAX_POSITIONS - len(positions)

    if pending_entries and available_slots > 0:
      all_valid_candidates = [
          item
          for item in pending_entries
          if item['ticker'] in week_data.index
          and item['ticker'] not in positions
      ]
      valid_candidates = all_valid_candidates[:available_slots]
      num_trades = len(valid_candidates)

      if num_trades > 0 and cash > 0:
        target_slot_capital = monday_total_equity / MAX_POSITIONS
        capital_per_trade = min(target_slot_capital, cash / num_trades)

        for item in valid_candidates:
          ticker = item['ticker']
          row = week_data.loc[ticker]
          raw_open = row['Open']

          exec_buy_price = raw_open * (1 + SLIPPAGE_PCT)
          effective_cost_per_unit = exec_buy_price + (
              raw_open * TRANSACTION_FEE_PCT
          )

          if effective_cost_per_unit > 0:
            alloc = min(capital_per_trade, cash)
            units = math.floor(alloc / effective_cost_per_unit)

            if units > 0:
              gross_buy_cost = units * exec_buy_price
              raw_trade_value = units * raw_open
              buy_brokerage = raw_trade_value * TRANSACTION_FEE_PCT
              actual_cost = gross_buy_cost + buy_brokerage

              if actual_cost <= cash:
                cash -= actual_cost
                total_brokerage_paid += buy_brokerage

                positions[ticker] = {
                    'units': units,
                    'buy_price': exec_buy_price,
                    'total_cost': actual_cost,
                    'buy_date': current_date,
                    'signal_date': item['signal_date'],
                    'rsi_signal': item['rsi_signal'],
                    'st_signal': item['st_signal'],
                    'rs_signal': item['rs_signal'],
                }

    pending_entries = []  # Reset outside conditional (Prevents stale signals)

    # -------------------------------------------------------------------
    # STEP 3: SCAN SIGNALS AT FRIDAY CLOSE (For Next Week Execution)
    # -------------------------------------------------------------------
    # Don't scan new entries on the final week of the block (liquidating)
    if idx < len(sorted_dates) - 1:
        # Get Nifty status for current Friday (fallback to True if missing date)
        lookup_date = pd.Timestamp(current_date).normalize()
        nifty_is_bullish = nifty_regime_dict.get(lookup_date, False)

        # 3A. Individual Stock Exit Signals
        for ticker, pos in positions.items():
            if ticker in week_data.index:
                row = week_data.loc[ticker]

                # Exit if stock hits RSI/ST stop OR if Nifty breaks below 200 SMA
                stock_signal_exit = (row['RSI'] < 40) or (row['ST_Direction'] == -1)
                market_regime_exit = not nifty_is_bullish

                if stock_signal_exit or market_regime_exit:
                    if ticker not in [x['ticker'] for x in pending_exits]:
                        pending_exits.append({
                            'ticker': ticker,
                            'exit_signal_date': current_date,
                            'exit_rsi': row['RSI'],
                            'exit_st': row['ST_Direction'],
                        })

        # 3B. New Entry Signals (ONLY if Nifty is above 200 SMA)
        tickers_exiting_next = [x['ticker'] for x in pending_exits]
        estimated_free_slots = MAX_POSITIONS - (len(positions) - len(pending_exits))

        if nifty_is_bullish and estimated_free_slots > 0:
            candidates = week_data[
                (week_data['RSI'] > 60)
                & (week_data['ST_Direction'] == 1)
                & (week_data['RS_Percentile'] >= 92)
                & (~week_data.index.isin(positions.keys()))
                & (~week_data.index.isin(tickers_exiting_next))
            ]

            if not candidates.empty:
                sorted_candidates = candidates.sort_values(
                    by=['RS_Percentile', 'RSI'], ascending=[False, False]
                )
                for tkr, row in sorted_candidates.iterrows():
                    pending_entries.append({
                        'ticker': tkr,
                        'signal_date': current_date,
                        'rsi_signal': row['RSI'],
                        'st_signal': row['ST_Direction'],
                        'rs_signal': row['RS_Percentile'],
                    })

    # -------------------------------------------------------------------
    # STEP 4: RECORD END-OF-WEEK EQUITY
    # -------------------------------------------------------------------
    friday_invested_value = sum(
        pos['units']
        * (
            week_data.loc[tkr, 'Close']
            if tkr in week_data.index
            else prev_week_closes.get(tkr, pos['buy_price'])
        )
        for tkr, pos in positions.items()
    )
    equity_history.append(cash + friday_invested_value)
    dates_history.append(current_date)

    for tkr in week_data.index:
      prev_week_closes[tkr] = week_data.loc[tkr, 'Close']

  # -------------------------------------------------------------------
  # BLOCK END: FORCE LIQUIDATE ALL OPEN POSITIONS TO CASH
  # -------------------------------------------------------------------
  final_week_data = weekly_df[
      weekly_df['Date'] == sorted_dates[-1]
  ].set_index('Ticker')

  for tkr, pos in list(positions.items()):
    last_close = (
        final_week_data.loc[tkr, 'Close']
        if tkr in final_week_data.index
        else pos['buy_price']
    )
    exec_sell_price = last_close * (1 - SLIPPAGE_PCT)
    gross_proceeds = pos['units'] * exec_sell_price

    raw_trade_value = pos['units'] * last_close
    sell_brokerage = raw_trade_value * TRANSACTION_FEE_PCT
    net_proceeds = gross_proceeds - sell_brokerage

    cash += net_proceeds
    total_brokerage_paid += sell_brokerage

    net_pnl_pct = (
        (net_proceeds - pos['total_cost']) / pos['total_cost']
    ) * 100

    completed_trades_log.append({
        'Block': f'{active_start}-{active_end}',
        'Ticker': tkr,
        'Signal_Date': pos['signal_date'].strftime('%Y-%m-%d'),
        'Buy_Date': pos['buy_date'].strftime('%Y-%m-%d'),
        'Buy_Price': round(pos['buy_price'], 2),
        'Units': pos['units'],
        'Entry_RSI': round(pos['rsi_signal'], 2),
        'Entry_ST': pos['st_signal'],
        'Signal_RS_Score': round(pos['rs_signal'], 2),
        'Exit_Signal_Date': sorted_dates[-1].strftime('%Y-%m-%d'),
        'Sell_Date': sorted_dates[-1].strftime('%Y-%m-%d'),
        'Sell_Price': round(exec_sell_price, 2),
        'Exit_RSI': 0.0,
        'Exit_ST': 0,
        'Return_%': round(net_pnl_pct, 2),
        'Exit_Reason': '2-Year Block Forced Liquidation',
    })

  positions.clear()
  current_capital = cash
  print(
      f'Block {active_start}-{active_end} Finished | Ending Cash:'
      f' Rs.{current_capital:,.2f}\n'
  )

# ==========================================
# EXPORT LOGS AND PERFORMANCE SUMMARY
# ==========================================
trades_df = pd.DataFrame(completed_trades_log)
portfolio_df = pd.DataFrame({'Date': dates_history, 'Equity': equity_history})

trades_df.to_csv(TRADES_LOG_OUTPUT, index=False)
portfolio_df.to_csv(EQUITY_CURVE_OUTPUT, index=False)

if not portfolio_df.empty:
  final_capital = portfolio_df['Equity'].iloc[-1]
  total_return_pct = (
      (final_capital - INITIAL_CAPITAL) / INITIAL_CAPITAL
  ) * 100

  start_date = pd.to_datetime(portfolio_df['Date'].iloc[0])
  end_date = pd.to_datetime(portfolio_df['Date'].iloc[-1])
  total_years = (end_date - start_date).days / 365.25

  cagr_pct = (
      (((final_capital / INITIAL_CAPITAL) ** (1 / total_years)) - 1) * 100
      if total_years > 0
      else 0.0
  )
  avg_annual_brokerage = (
      total_brokerage_paid / total_years if total_years > 0 else 0.0
  )

  # Global Max Drawdown Calculation
  cum_max = portfolio_df['Equity'].cummax()
  portfolio_df['Drawdown_%'] = (
      (portfolio_df['Equity'] - cum_max) / cum_max
  ) * 100
  max_drawdown_pct = portfolio_df['Drawdown_%'].min()

  # Sharpe Ratio Calculation (Annualized Weekly)
  portfolio_df['Weekly_Return'] = portfolio_df['Equity'].pct_change()
  weekly_returns = portfolio_df['Weekly_Return'].dropna()

  rf_weekly = (1 + RISK_FREE_RATE_ANNUAL) ** (1 / 52) - 1
  excess_weekly_returns = weekly_returns - rf_weekly

  std_weekly = weekly_returns.std(ddof=1)
  sharpe_ratio = (
      (excess_weekly_returns.mean() / std_weekly) * math.sqrt(52)
      if std_weekly > 0
      else 0.0
  )

  total_trades = len(trades_df)
  win_rate_pct = (
      (len(trades_df[trades_df['Return_%'] > 0]) / total_trades) * 100
      if total_trades > 0
      else 0.0
  )

  portfolio_df['Year'] = pd.to_datetime(portfolio_df['Date']).dt.year
  yearly_returns = []
  yearly_drawdowns = []
  unique_years = portfolio_df['Year'].unique()

  for i, yr in enumerate(unique_years):
    group = portfolio_df[portfolio_df['Year'] == yr]
    start_val = (
        INITIAL_CAPITAL
        if i == 0
        else portfolio_df[portfolio_df['Year'] == unique_years[i - 1]][
            'Equity'
        ].iloc[-1]
    )
    end_val = group['Equity'].iloc[-1]

    y_return = ((end_val - start_val) / start_val) * 100
    yearly_returns.append(y_return)
    yearly_drawdowns.append(group['Drawdown_%'].min())

  avg_annual_return_pct = pd.Series(yearly_returns).mean()
  avg_annual_drawdown_pct = pd.Series(yearly_drawdowns).mean()

  print('=' * 60)
  print('          MULTI-BLOCK OVERALL STRATEGY PERFORMANCE')
  print('=' * 60)
  print(f'Initial Capital        : Rs.{INITIAL_CAPITAL:,.2f}')
  print(f'Final Capital          : Rs.{final_capital:,.2f}')
  print(f'Total Return           : {total_return_pct:+.2f}%')
  print(f'CAGR                   : {cagr_pct:.2f}%')
  print(
      'Sharpe Ratio'
      f' (R_f={RISK_FREE_RATE_ANNUAL*100:.1f}%): {sharpe_ratio:.2f}'
  )
  print(f'Max Drawdown           : {max_drawdown_pct:.2f}%')
  print(f'Total Trades           : {total_trades}')
  print(f'Win Rate               : {win_rate_pct:.2f}%')
  print(f'Total Brokerage Paid   : Rs.{total_brokerage_paid:,.2f}')
  print(f'Avg Annual Brokerage   : Rs.{avg_annual_brokerage:,.2f}')
  print(f'Avg Annual Return      : {avg_annual_return_pct:+.2f}%')
  print(f'Avg Annual Drawdown    : {avg_annual_drawdown_pct:.2f}%')
  print('=' * 60)

Starting Backtest across 4 2-year blocks...

--- Running Block 2017-2018 | Starting Cash: Rs.1,000,000.00 ---
Block 2017-2018 Finished | Ending Cash: Rs.1,081,798.33

--- Running Block 2019-2020 | Starting Cash: Rs.1,081,798.33 ---
Block 2019-2020 Finished | Ending Cash: Rs.1,055,170.89

--- Running Block 2021-2022 | Starting Cash: Rs.1,055,170.89 ---
Block 2021-2022 Finished | Ending Cash: Rs.1,958,536.00

--- Running Block 2023-2024 | Starting Cash: Rs.1,958,536.00 ---
Block 2023-2024 Finished | Ending Cash: Rs.4,359,532.35

          MULTI-BLOCK OVERALL STRATEGY PERFORMANCE
Initial Capital        : Rs.1,000,000.00
Final Capital          : Rs.4,359,532.35
Total Return           : +335.95%
CAGR                   : 20.23%
Sharpe Ratio (R_f=0.0%): 0.93
Max Drawdown           : -53.32%
Total Trades           : 320
Win Rate               : 38.75%
Total Brokerage Paid   : Rs.284,220.75
Avg Annual Brokerage   : Rs.35,564.11
Avg Annual Return      : +25.02%
Avg Annual Drawdown    : -30.68%


In [ ]:
import math
import pandas as pd

# ==========================================
# FILE PATHS FOR BOTH RUNS
# ==========================================
EQUITY_OLD_PATH = '/content/drive/MyDrive/Stock_Backtest/3_backtest_results/equity_curve.csv'
TRADES_OLD_PATH = '/content/drive/MyDrive/Stock_Backtest/3_backtest_results/completed_trades_log.csv'

EQUITY_NEW_PATH = 'equity_curve (1).csv'
TRADES_NEW_PATH = 'completed_trades_log (3).csv'

INITIAL_CAPITAL_2000 = 10_00_000.0  # Original starting cash in 2000
RISK_FREE_RATE_ANNUAL = 0.00

# ==========================================
# 1. COMBINE EQUITY CURVES & TRADES
# ==========================================
df_eq_old = pd.read_csv(EQUITY_OLD_PATH)
df_eq_new = pd.read_csv(EQUITY_NEW_PATH)

# Combine and drop potential overlapping date at boundary
df_equity = pd.concat([df_eq_old, df_eq_new], ignore_index=True)
df_equity['Date'] = pd.to_datetime(df_equity['Date'])
df_equity = df_equity.drop_duplicates(subset=['Date'], keep='last').sort_values('Date').reset_index(drop=True)

df_tr_old = pd.read_csv(TRADES_OLD_PATH)
df_tr_new = pd.read_csv(TRADES_NEW_PATH)
df_trades = pd.concat([df_tr_old, df_tr_new], ignore_index=True)

# ==========================================
# 2. RECALCULATE UNIFIED GLOBAL METRICS
# ==========================================
final_capital = df_equity['Equity'].iloc[-1]
total_return_pct = ((final_capital - INITIAL_CAPITAL_2000) / INITIAL_CAPITAL_2000) * 100

start_date = df_equity['Date'].iloc[0]
end_date = df_equity['Date'].iloc[-1]
total_years = (end_date - start_date).days / 365.25

cagr_pct = (((final_capital / INITIAL_CAPITAL_2000) ** (1 / total_years)) - 1) * 100

# Global Max Drawdown
cum_max = df_equity['Equity'].cummax()
df_equity['Drawdown_%'] = ((df_equity['Equity'] - cum_max) / cum_max) * 100
max_drawdown_pct = df_equity['Drawdown_%'].min()

# Unified Sharpe Ratio
df_equity['Weekly_Return'] = df_equity['Equity'].pct_change()
weekly_returns = df_equity['Weekly_Return'].dropna()
rf_weekly = (1 + RISK_FREE_RATE_ANNUAL) ** (1 / 52) - 1
excess_returns = weekly_returns - rf_weekly

std_weekly = weekly_returns.std(ddof=1)
sharpe_ratio = (excess_returns.mean() / std_weekly) * math.sqrt(52) if std_weekly > 0 else 0.0

# Unified Trade Statistics
total_trades = len(df_trades)
winning_trades = len(df_trades[df_trades['Return_%'] > 0])
win_rate_pct = (winning_trades / total_trades) * 100 if total_trades > 0 else 0.0
df_equity.to_csv('equity_curve.csv',index=True)

# ==========================================
# 3. PRINT UNIFIED PERFORMANCE
# ==========================================
print("=" * 60)
print(f" UNIFIED OVERALL PERFORMANCE ({start_date.strftime('%Y')} - {end_date.strftime('%Y')})")
print("=" * 60)
print(f"Initial Capital (2000) : Rs.{INITIAL_CAPITAL_2000:,.2f}")
print(f"Current Final Capital  : Rs.{final_capital:,.2f}")
print(f"Total Return           : {total_return_pct:+.2f}%")
print(f"Overall CAGR           : {cagr_pct:.2f}%")
print(f"Unified Sharpe Ratio   : {sharpe_ratio:.2f}")
print(f"Global Max Drawdown    : {max_drawdown_pct:.2f}%")
print(f"Total Trades           : {total_trades}")
print(f"Overall Win Rate       : {win_rate_pct:.2f}%")
print("=" * 60)

 UNIFIED OVERALL PERFORMANCE (2017 - 2026)
Initial Capital (2000) : Rs.1,000,000.00
Current Final Capital  : Rs.6,509,327.04
Total Return           : +550.93%
Overall CAGR           : 21.65%
Unified Sharpe Ratio   : 0.72
Global Max Drawdown    : -53.32%
Total Trades           : 384
Overall Win Rate       : 37.50%
